In [ ]:
from google.colab import files
print("Upload cats_dogs_100.zip")
uploaded = files.upload() # choose cats_dogs_100.zip

Upload cats_dogs_100.zip


TypeError: 'NoneType' object is not subscriptable

In [ ]:
import os, zipfile
zip_name = "cats_dogs_100.zip"
# Unzip into current directory (this will create a folder "content/cats_dogs_100")
with zipfile.ZipFile(zip_name, "r") as zip_ref:
    zip_ref.extractall(".")
print("Root contents:", os.listdir())
print("Inside 'content':", os.listdir("content"))
print("Inside 'content/cats_dogs_100':", os.listdir("content/cats_dogs_100"))

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import cv2
import os
img_size = (128, 128)
batch_size = 16
train_dir = "content/cats_dogs_100/train"
val_dir = "content/cats_dogs_100/val"
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
 train_dir,
 image_size=img_size,
 batch_size=batch_size
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
 val_dir,
 image_size=img_size,
 batch_size=batch_size
)
class_names = train_ds.class_names
print("Classes:", class_names)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

In [ ]:
plt.figure(figsize=(8, 8))
for images, labels in train_ds.take(1):
 for i in range(9):
  ax = plt.subplot(3, 3, i + 1)
 plt.imshow(images[i].numpy().astype("uint8"))
 plt.title(class_names[labels[i]])
 plt.axis("off")

In [ ]:
inputs = layers.Input(shape=img_size + (3,))
x = layers.Rescaling(1./255)(inputs)
x = layers.Conv2D(16, (3, 3), activation="relu")(x)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(32, (3, 3), activation="relu")(x)
x = layers.MaxPooling2D()(x)
x = layers.Conv2D(64, (3, 3), activation="relu", name="last_conv")(x)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(2, activation="softmax")(x)
model = models.Model(inputs=inputs, outputs=outputs)
model.compile(
 optimizer="adam",
 loss="sparse_categorical_crossentropy",
 metrics=["accuracy"]
)
model.summary()


In [ ]:
history = model.fit(
train_ds,
validation_data=val_ds,
epochs=15, # you can change to 5 if time is short
verbose=1
)


In [ ]:
val_images, val_labels = next(iter(val_ds))
idx = 0 # change to any index from 0 to 15 to test different images within the batch
image = val_images[idx:idx+1]
true_label = val_labels[idx].numpy()
preds = model.predict(image)
pred_class = np.argmax(preds[0])
print("True:", class_names[true_label])
print("Pred:", class_names[pred_class], "(prob =", preds[0][pred_class], ")")
plt.imshow(val_images[idx].numpy().astype("uint8"))
plt.title(f"True: {class_names[true_label]}, Pred: {class_names[pred_class]}")
plt.axis("off")
plt.show()

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
 # Get last conv layer
 last_conv_layer = model.get_layer(last_conv_layer_name)
 # Build a model that maps input -> (last conv layer output, predictions)
 heatmap_model = tf.keras.models.Model(
 [model.input],
 [last_conv_layer.output, model.output]
 )
 with tf.GradientTape() as tape:
  conv_outputs, predictions = heatmap_model(img_array)
  if pred_index is None:
   pred_index = tf.argmax(predictions[0])
  class_channel = predictions[:, pred_index]
  # Gradient of class score w.r.t. conv feature map
  grads = tape.gradient(class_channel, conv_outputs)
  # Global average pooling over H,W
  pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
  conv_outputs = conv_outputs[0] # (H, W, C)
  heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
  heatmap = tf.squeeze(heatmap)
  # ReLU + normalize to [0,1]
  heatmap = tf.maximum(heatmap, 0)
  heatmap /= tf.math.reduce_max(heatmap + 1e-8)
 return heatmap.numpy()

In [ ]:
# Create heatmap
heatmap = make_gradcam_heatmap(
 image,
 model,
 last_conv_layer_name="last_conv"
)
plt.imshow(heatmap)
plt.title("Grad-CAM heatmap")
plt.axis("off")
plt.show()
def overlay_heatmap_on_image(heatmap, img, alpha=0.4):
 # Resize heatmap to image size
 heatmap = cv2.resize(heatmap, (img.shape[1], img.shape[0]))
 heatmap = np.uint8(255 * heatmap)
 # Color map
 heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)
 # Overlay
 overlay = heatmap_color * alpha + img
 return np.uint8(overlay)
img_np = val_images[idx].numpy().astype("uint8")
overlay = overlay_heatmap_on_image(heatmap, img_np)
plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
plt.imshow(img_np)
plt.title("Original")
plt.axis("off")
plt.subplot(1, 2, 2)
plt.imshow(overlay)
plt.title("Grad-CAM Overlay")
plt.axis("off")
plt.show()